# IAN 603 · Module 10
# Reshaping Data in PostgreSQL

This notebook is the hands-on, step-by-step companion for **Module 10: Reshaping Data in PostgreSQL**.

The central idea is:

> **Change the structure of the data without accidentally changing the meaning of the underlying facts.**

### Main topics
- Long vs. wide data
- Grain
- Pivoting with `CASE`
- Pivoting with `FILTER`
- Unpivoting with `UNION ALL`
- Unpivoting with `CROSS JOIN LATERAL ... VALUES`
- NULL vs. zero
- Reversible vs. lossy reshaping
- Pre-aggregation
- Validation
- Choosing the right shape for the next task

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module10.html



## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the difference between long and wide data.
2. State the **grain** before and after a reshape.
3. Pivot long data into columns using conditional aggregation.
4. Explain when `CASE` and `FILTER` are equivalent.
5. Preserve the difference between missing values and zeros.
6. Unpivot wide data into long form with `UNION ALL`.
7. Unpivot compactly with `LATERAL VALUES`.
8. Explain when a reshape is reversible and when information is lost.
9. Pre-aggregate before reshaping when one target cell represents several source rows.
10. Validate totals, row counts, and categories before accepting a reshape.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Build the Module 10 Teaching Database

Module 10 uses several small tables so we can reshape the same kinds of facts in different ways.

| Table | Grain |
|---|---|
| `patients` | one row per patient |
| `admissions` | one row per admission |
| `sales` | one row per sales transaction |
| `survey_responses` | one row per respondent-question |
| `monthly_sales` | one row per product |
| `patient_metrics` | one row per patient |
| `students` | one row per student |
| `grades` | one row per student-course |

The important habit is to always ask:

> **What does one row represent before the reshape, and what should one row represent after it?**


## 4. Create or reset all Module 10 tables

This setup is repeatable. Re-run it whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 10\n-- Reshaping Data in PostgreSQL\n-- ============================================================\n-- Companion file for:\n-- s_module10.html\n--\n-- Designed for PostgreSQL and pgAdmin Query Tool.\n--\n-- This file is intentionally more detailed than the slides.\n-- It includes:\n--   1. Teaching database setup\n--   2. Table/grain explanations\n--   3. Pivot examples\n--   4. Unpivot examples\n--   5. NULL and grain checks\n--   6. Validation queries\n--   7. Module practice queries\n--   8. Additional optional practice\n--\n-- HOW TO USE\n-- ------------------------------------------------------------\n-- 1. Open pgAdmin.\n-- 2. Open Query Tool for your practice database.\n-- 3. Run SECTION 1 once to create/reset the teaching tables.\n-- 4. Run later examples one at a time while following Module 10.\n-- 5. Re-run SECTION 1 whenever you want a clean starting point.\n--\n-- NOTE\n-- ------------------------------------------------------------\n-- SECTION 1 uses DROP TABLE IF EXISTS. It is intended for a\n-- practice database, not a production database.\n-- ============================================================\n\n\n-- ============================================================\n-- TABLE GUIDE\n-- ============================================================\n--\n-- patients\n--   Grain: one row per patient.\n--   Used to demonstrate preserving all patients with LEFT JOIN.\n--\n-- admissions\n--   Grain: one row per hospital admission.\n--   Useful dimensions: diagnosis, department, admit_time.\n--   Useful measure: admission count.\n--\n-- sales\n--   Grain: one row per sales transaction.\n--   Useful dimensions: product, category, sale_month.\n--   Measure: revenue.\n--   Important: several transactions can belong to the same\n--   product-month, so aggregation is required before/pivoting.\n--\n-- survey_responses\n--   Grain: one row per respondent-question.\n--   Used for converting question values into separate columns.\n--\n-- monthly_sales\n--   Grain: one row per product.\n--   Jan/Feb/Mar are stored as separate columns, so this table\n--   begins in wide form and is useful for unpivot examples.\n--\n-- patient_metrics\n--   Grain: one row per patient.\n--   Several measurements are stored in columns. This is another\n--   wide-form table used for LATERAL VALUES unpivot examples.\n--\n-- students\n--   Grain: one row per student.\n--\n-- grades\n--   Grain: one row per student-course.\n--   Used for building a grade matrix with course columns.\n--\n-- Key idea:\n--   Always state the source grain and target grain before writing\n--   a reshape query. A pivot often changes the row grain.\n\n\n\n-- ============================================================\n-- QUICK CONCEPT NOTES\n-- ============================================================\n--\n-- LONG FORM\n--   Repeated categories are stored as row values.\n--   Example:\n--     product | sale_month | revenue\n--\n-- WIDE FORM\n--   Selected category values become columns.\n--   Example:\n--     product | jan_revenue | feb_revenue | mar_revenue\n--\n-- PIVOT IN POSTGRESQL\n--   PostgreSQL does not have a core PIVOT keyword.\n--   In this module, use conditional aggregation:\n--\n--     SUM(revenue) FILTER (WHERE condition)\n--\n--   or:\n--\n--     SUM(CASE WHEN condition THEN revenue END)\n--\n-- UNPIVOT IN POSTGRESQL\n--   PostgreSQL does not have a core UNPIVOT keyword.\n--   Common patterns:\n--     1. UNION ALL\n--     2. CROSS JOIN LATERAL with VALUES\n--\n-- NULL VS ZERO\n--   NULL often means no recorded observation.\n--   Zero means a real observed value of zero.\n--   Do not replace NULL with zero unless the business meaning\n--   supports that interpretation.\n--\n-- VALIDATION\n--   After reshaping, compare:\n--     * row grain\n--     * row counts\n--     * total measures\n--     * expected categories\n--     * NULL behavior\n\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET THE MODULE 10 TEACHING DATABASE\n-- ============================================================\n-- The commands below recreate the same small datasets used by\n-- the interactive Module 10 page.\n-- ============================================================\n\nDROP TABLE IF EXISTS grades CASCADE;\nDROP TABLE IF EXISTS students CASCADE;\nDROP TABLE IF EXISTS patient_metrics CASCADE;\nDROP TABLE IF EXISTS monthly_sales CASCADE;\nDROP TABLE IF EXISTS survey_responses CASCADE;\nDROP TABLE IF EXISTS sales CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n  subject_id integer PRIMARY KEY,\n  name text NOT NULL\n);\n\nCREATE TABLE admissions (\n  hadm_id integer PRIMARY KEY,\n  subject_id integer NOT NULL REFERENCES patients(subject_id),\n  admit_time timestamp NOT NULL,\n  diagnosis text NOT NULL,\n  department text NOT NULL\n);\n\nCREATE TABLE sales (\n  sale_id integer PRIMARY KEY,\n  product text NOT NULL,\n  category text NOT NULL,\n  sale_month date NOT NULL,\n  revenue numeric(10,2) NOT NULL CHECK (revenue >= 0)\n);\n\nCREATE TABLE survey_responses (\n  respondent_id integer NOT NULL,\n  question text NOT NULL,\n  score integer CHECK (score BETWEEN 1 AND 5),\n  PRIMARY KEY (respondent_id, question)\n);\n\nCREATE TABLE monthly_sales (\n  product text PRIMARY KEY,\n  jan_revenue numeric(10,2),\n  feb_revenue numeric(10,2),\n  mar_revenue numeric(10,2)\n);\n\nCREATE TABLE patient_metrics (\n  patient_id integer PRIMARY KEY,\n  heart_rate integer,\n  systolic_bp integer,\n  diastolic_bp integer,\n  oxygen_sat numeric(5,2)\n);\n\nCREATE TABLE students (\n  student_id integer PRIMARY KEY,\n  student_name text NOT NULL\n);\n\nCREATE TABLE grades (\n  student_id integer REFERENCES students(student_id),\n  course_code text NOT NULL,\n  grade numeric(5,2),\n  PRIMARY KEY (student_id, course_code)\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith'),\n(1002,'Bob Johnson'),\n(1003,'Carol White'),\n(1004,'David Brown'),\n(1005,'Eve Davis');\n\nINSERT INTO admissions VALUES\n(101,1001,TIMESTAMP '2025-01-10 09:00','Pneumonia','Internal Medicine'),\n(102,1002,TIMESTAMP '2025-01-15 14:00','Fracture','Orthopedics'),\n(103,1001,TIMESTAMP '2025-01-28 16:00','Flu','Internal Medicine'),\n(104,1001,TIMESTAMP '2025-02-05 08:30','Pneumonia','Internal Medicine'),\n(105,1003,TIMESTAMP '2025-02-10 11:00','Flu','Internal Medicine'),\n(106,1004,TIMESTAMP '2025-02-18 07:45','Pneumonia','Pulmonology'),\n(107,1005,TIMESTAMP '2025-03-02 13:20','Fracture','Orthopedics'),\n(108,1002,TIMESTAMP '2025-03-15 19:10','Flu','Internal Medicine'),\n(109,1001,TIMESTAMP '2025-03-22 10:15','Flu','Internal Medicine'),\n(110,1003,TIMESTAMP '2025-03-30 17:00','Pneumonia','Pulmonology');\n\nINSERT INTO sales VALUES\n(1,'Laptop','Electronics',DATE '2025-01-01',900.00),\n(2,'Laptop','Electronics',DATE '2025-01-01',700.00),\n(3,'Mouse','Accessories',DATE '2025-01-01',40.00),\n(4,'Mouse','Accessories',DATE '2025-01-01',60.00),\n(5,'Laptop','Electronics',DATE '2025-02-01',1400.00),\n(6,'Laptop','Electronics',DATE '2025-02-01',1000.00),\n(7,'Mouse','Accessories',DATE '2025-02-01',150.00),\n(8,'Keyboard','Accessories',DATE '2025-02-01',200.00),\n(9,'Laptop','Electronics',DATE '2025-03-01',1800.00),\n(10,'Mouse','Accessories',DATE '2025-03-01',190.00),\n(11,'Keyboard','Accessories',DATE '2025-03-01',250.00);\n\nINSERT INTO survey_responses VALUES\n(1,'Q1',4),(1,'Q2',5),(1,'Q3',4),\n(2,'Q1',3),(2,'Q2',4),\n(3,'Q1',5),(3,'Q2',5),(3,'Q3',5);\n\nINSERT INTO monthly_sales VALUES\n('Laptop',1600.00,2400.00,1800.00),\n('Mouse',100.00,150.00,190.00),\n('Keyboard',NULL,200.00,250.00);\n\nINSERT INTO patient_metrics VALUES\n(1001,72,120,80,98.0),\n(1002,68,130,85,97.0),\n(1003,88,142,92,94.5);\n\nINSERT INTO students VALUES\n(1,'Ana Patel'),(2,'Malik Green'),(3,'Sofia Chen');\n\nINSERT INTO grades VALUES\n(1,'CSC101',92),(1,'STA201',88),(1,'DBA301',95),\n(2,'CSC101',84),(2,'STA201',91),\n(3,'CSC101',97),(3,'STA201',94),(3,'DBA301',90);\n\n\n"

execute_script(setup_sql)


## 5. Confirm the teaching tables loaded


In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM patients) AS patients,
    (SELECT COUNT(*) FROM admissions) AS admissions,
    (SELECT COUNT(*) FROM sales) AS sales,
    (SELECT COUNT(*) FROM survey_responses) AS survey_responses,
    (SELECT COUNT(*) FROM monthly_sales) AS monthly_sales,
    (SELECT COUNT(*) FROM patient_metrics) AS patient_metrics,
    (SELECT COUNT(*) FROM students) AS students,
    (SELECT COUNT(*) FROM grades) AS grades;
""", title="Module 10 row counts")


# Part II · Long vs. Wide Data

The same facts can often be represented in different shapes.

### Long form

| product | month | revenue |
|---|---|---:|
| Laptop | Jan | 1200 |
| Laptop | Feb | 1500 |
| Mouse | Jan | 300 |
| Mouse | Feb | 420 |

Grain:

> **one row per product-month**

### Wide form

| product | Jan | Feb |
|---|---:|---:|
| Laptop | 1200 | 1500 |
| Mouse | 300 | 420 |

Grain:

> **one row per product**

The values may represent the same information, but the row meaning has changed.


## 6. Inspect a long sales structure


In [ ]:
display_query("""
SELECT
    sale_id,
    product,
    category,
    sale_month,
    revenue
FROM sales
ORDER BY product, sale_month, sale_id;
""", title="Long transaction data")


### Important: transaction grain is even more detailed

The raw `sales` table is:

> **one row per transaction**

Before a clean pivot to one product-month cell, several transaction rows may need to be combined.


## 7. Summarize to product-month grain first


In [ ]:
display_query("""
SELECT
    product,
    sale_month,
    COUNT(*) AS source_rows,
    SUM(revenue) AS revenue
FROM sales
GROUP BY product, sale_month
ORDER BY product, sale_month;
""", title="One row per product-month")


The output is now a true long-form analytical table:

| product | month | revenue |
|---|---|---:|
| Laptop | Jan | combined Jan revenue |
| Laptop | Feb | combined Feb revenue |
| Mouse | Jan | combined Jan revenue |

This is often the safest input for a pivot.


## 8. Inspect a wide source


In [ ]:
display_query("""
SELECT *
FROM monthly_sales
ORDER BY product;
""", title="Wide monthly sales")


# Part III · Grain: the Safeguard

Reshaping is not just moving values around.

It often changes the output grain.

### Example

Raw sales:

| sale_id | product | month | revenue |
|---:|---|---|---:|
| 1 | Laptop | Jan | 500 |
| 2 | Laptop | Jan | 700 |

Source grain:

> one row per transaction

If we want one January cell for Laptop, both source rows map to the same target cell.

That means we need an aggregation rule:

```text
500 + 700 = 1200
```

So the pivot is not only reshaping. It is also **aggregating**.


## 9. Check whether target cells have multiple source rows

If `source_rows > 1`, a pivot cell needs a rule such as `SUM`, `COUNT`, `MAX`, or `AVG`.


In [ ]:
display_query("""
SELECT
    product,
    sale_month,
    COUNT(*) AS source_rows,
    SUM(revenue) AS total_revenue
FROM sales
GROUP BY product, sale_month
ORDER BY product, sale_month;
""", title="Source cardinality by future pivot cell")


# Part IV · Pivoting Long Data to Wide Data

A pivot usually means:

```text
row categories
    ↓
become columns
```

For example:

**Long**

| month | diagnosis |
|---|---|
| Jan | Pneumonia |
| Jan | Flu |
| Feb | Fracture |

**Wide**

| month | pneumonia | fracture | flu |
|---|---:|---:|---:|
| Jan | 1 | 0 | 1 |
| Feb | 0 | 1 | 0 |

PostgreSQL can do this cleanly with **conditional aggregation**.


## 10. Pivot method 1: `SUM(CASE ...)`

The pattern is:

```sql
SUM(
    CASE
        WHEN condition THEN value
        ELSE 0
    END
)
```

For counts, the value can simply be 1.


In [ ]:
display_query("""
SELECT
    DATE_TRUNC('month', admit_time)::date AS admit_month,

    SUM(
        CASE WHEN diagnosis = 'Pneumonia' THEN 1 ELSE 0 END
    ) AS pneumonia_count,

    SUM(
        CASE WHEN diagnosis = 'Fracture' THEN 1 ELSE 0 END
    ) AS fracture_count,

    SUM(
        CASE WHEN diagnosis = 'Flu' THEN 1 ELSE 0 END
    ) AS flu_count

FROM admissions
GROUP BY 1
ORDER BY 1;
""", title="Diagnosis pivot with CASE")


### Read the CASE pivot one row at a time

Suppose one month has:

| diagnosis |
|---|
| Pneumonia |
| Flu |
| Pneumonia |

For the Pneumonia expression:

| diagnosis | CASE result |
|---|---:|
| Pneumonia | 1 |
| Flu | 0 |
| Pneumonia | 1 |

Then:

```text
SUM = 2
```

Each output column applies a different condition.


## 11. Pivot method 2: `FILTER`

PostgreSQL provides a very readable alternative:

```sql
COUNT(*) FILTER (WHERE condition)
```


In [ ]:
display_query("""
SELECT
    DATE_TRUNC('month', admit_time)::date AS admit_month,
    COUNT(*) FILTER (
        WHERE diagnosis = 'Pneumonia'
    ) AS pneumonia_count,
    COUNT(*) FILTER (
        WHERE diagnosis = 'Fracture'
    ) AS fracture_count,
    COUNT(*) FILTER (
        WHERE diagnosis = 'Flu'
    ) AS flu_count
FROM admissions
GROUP BY 1
ORDER BY 1;
""", title="Diagnosis pivot with FILTER")


## 12. CASE vs. FILTER

For many PostgreSQL pivots, both are valid.

| Style | Strength |
|---|---|
| `SUM(CASE ...)` | portable and very flexible |
| `COUNT/SUM(...) FILTER (...)` | often shorter and easier to read in PostgreSQL |

The important decision is not syntax preference.

The important decisions are:
- output grain,
- target categories,
- aggregation rule,
- NULL behavior.


## 13. Pivot sales revenue by product

Here the output grain is:

> **one row per month**


In [ ]:
display_query("""
SELECT
    sale_month,
    SUM(revenue) FILTER (
        WHERE product = 'Laptop'
    ) AS laptop_revenue,
    SUM(revenue) FILTER (
        WHERE product = 'Mouse'
    ) AS mouse_revenue,
    SUM(revenue) FILTER (
        WHERE product = 'Keyboard'
    ) AS keyboard_revenue
FROM sales
GROUP BY sale_month
ORDER BY sale_month;
""", title="Product revenue columns")


# Part V · Pivoting Survey Responses

Survey data naturally arrives in long form:

| respondent_id | question | score |
|---:|---|---:|
| 1 | Q1 | 4 |
| 1 | Q2 | 5 |
| 1 | Q3 | 3 |
| 2 | Q1 | 2 |
| 2 | Q2 | 4 |

For modeling or export, we may want:

| respondent_id | q1 | q2 | q3 |
|---:|---:|---:|---:|
| 1 | 4 | 5 | 3 |
| 2 | 2 | 4 | NULL |

The output grain becomes:

> **one row per respondent**


## 14. Pivot survey scores


In [ ]:
display_query("""
SELECT
    respondent_id,
    MAX(score) FILTER (
        WHERE question = 'Q1'
    ) AS q1_score,
    MAX(score) FILTER (
        WHERE question = 'Q2'
    ) AS q2_score,
    MAX(score) FILTER (
        WHERE question = 'Q3'
    ) AS q3_score
FROM survey_responses
GROUP BY respondent_id
ORDER BY respondent_id;
""", title="Survey responses in wide form")


### Why `MAX(score)`?

If the data model guarantees at most one score per respondent-question, `MAX` simply returns that value.

But if duplicates exist, `MAX` silently chooses one of them.

That is why **source grain validation comes before reshaping**.


# Part VI · NULL and Zero Are Not the Same

Suppose a respondent never answered Q3.

These two expressions mean different things:

```sql
MAX(CASE WHEN question = 'Q3' THEN score END)
```

and:

```sql
MAX(CASE WHEN question = 'Q3' THEN score ELSE 0 END)
```

### Preserve missing

| respondent | Q3 |
|---|---:|
| 2 | NULL |

Meaning:

> no Q3 observation exists

### Force zero

| respondent | Q3 |
|---|---:|
| 2 | 0 |

Meaning:

> a Q3 observation is represented as zero

Those are different facts.


## 15. Compare NULL preservation with forced zero


In [ ]:
display_query("""
SELECT
    respondent_id,
    MAX(
        CASE WHEN question = 'Q3' THEN score END
    ) AS q3_preserve_missing,
    MAX(
        CASE WHEN question = 'Q3' THEN score ELSE 0 END
    ) AS q3_force_zero
FROM survey_responses
GROUP BY respondent_id
ORDER BY respondent_id;
""", title="NULL vs zero")


### Rule of thumb

Do not replace `NULL` with zero unless zero truly has the same business meaning as “missing.”

Reshaping should preserve semantics, not just produce a neat table.


# Part VII · Unpivoting Wide Data to Long Data

Unpivoting moves repeated columns back into rows.

### Wide

| product | jan | feb | mar |
|---|---:|---:|---:|
| Laptop | 1200 | 1500 | 1400 |
| Mouse | 300 | 420 | NULL |

### Long

| product | month | revenue |
|---|---|---:|
| Laptop | Jan | 1200 |
| Laptop | Feb | 1500 |
| Laptop | Mar | 1400 |
| Mouse | Jan | 300 |
| Mouse | Feb | 420 |
| Mouse | Mar | NULL |

Source grain:

> one row per product

Target grain:

> one row per product-month


## 16. Method 1: `UNION ALL`

The most explicit method is to write one `SELECT` per source column.


In [ ]:
display_query("""
SELECT
    product,
    DATE '2025-01-01' AS sale_month,
    jan_revenue AS revenue
FROM monthly_sales

UNION ALL

SELECT
    product,
    DATE '2025-02-01',
    feb_revenue
FROM monthly_sales

UNION ALL

SELECT
    product,
    DATE '2025-03-01',
    mar_revenue
FROM monthly_sales

ORDER BY product, sale_month;
""", title="Unpivot with UNION ALL")


### Read the method as stacking tables

```text
January rows
    +
February rows
    +
March rows
    =
one long result
```

`UNION ALL` preserves all rows and is usually the correct stacking operation here.


## 17. Method 2: `LATERAL VALUES`

A more compact PostgreSQL pattern is:

```sql
CROSS JOIN LATERAL (
    VALUES
        (...)
        (...)
        (...)
) AS v(...)
```

For each wide source row, PostgreSQL generates several related long rows.


In [ ]:
display_query("""
SELECT
    m.product,
    v.sale_month,
    v.revenue
FROM monthly_sales AS m
CROSS JOIN LATERAL (
    VALUES
        (DATE '2025-01-01', m.jan_revenue),
        (DATE '2025-02-01', m.feb_revenue),
        (DATE '2025-03-01', m.mar_revenue)
) AS v(sale_month, revenue)
ORDER BY m.product, v.sale_month;
""", title="Unpivot with LATERAL VALUES")


### One source row becomes several rows

For:

| product | jan | feb | mar |
|---|---:|---:|---:|
| Laptop | 1200 | 1500 | 1400 |

the `VALUES` block generates:

| product | month | revenue |
|---|---|---:|
| Laptop | Jan | 1200 |
| Laptop | Feb | 1500 |
| Laptop | Mar | 1400 |

Then PostgreSQL repeats that logic for the next product.


## 18. UNION ALL vs. LATERAL VALUES

| Method | Advantage | Tradeoff |
|---|---|---|
| `UNION ALL` | very explicit and beginner-friendly | repetitive with many columns |
| `LATERAL VALUES` | compact and easy to extend | slightly more advanced syntax |

Both can produce the same output shape.


# Part VIII · Unpivoting Different Measures

Sometimes wide columns represent different metrics rather than months.

Example:

| patient_id | heart_rate | systolic_bp | diastolic_bp | oxygen_sat |
|---:|---:|---:|---:|---:|
| 1 | 72 | 120 | 80 | 98.5 |

Long form:

| patient_id | metric | value |
|---:|---|---:|
| 1 | heart_rate | 72 |
| 1 | systolic_bp | 120 |
| 1 | diastolic_bp | 80 |
| 1 | oxygen_sat | 98.5 |


## 19. Patient metrics to long form


In [ ]:
display_query("""
SELECT
    p.patient_id,
    v.metric,
    v.value
FROM patient_metrics AS p
CROSS JOIN LATERAL (
    VALUES
        ('heart_rate', p.heart_rate::numeric),
        ('systolic_bp', p.systolic_bp::numeric),
        ('diastolic_bp', p.diastolic_bp::numeric),
        ('oxygen_sat', p.oxygen_sat::numeric)
) AS v(metric, value)
ORDER BY p.patient_id, v.metric;
""", title="Patient metrics in long form")


### Why the casts?

Rows inside one `VALUES` column need compatible data types.

The `::numeric` casts standardize different numeric source types into one long `value` column.


# Part IX · Preserve or Drop NULL Cells

Unpivoting a wide table can create explicit rows for missing cells.

Suppose:

| product | Jan | Feb | Mar |
|---|---:|---:|---:|
| Mouse | 300 | 420 | NULL |

A full unpivot produces:

| product | month | revenue |
|---|---|---:|
| Mouse | Jan | 300 |
| Mouse | Feb | 420 |
| Mouse | Mar | NULL |

Whether that final NULL row should remain depends on the downstream meaning.


## 20. Remove NULL rows after unpivoting


In [ ]:
display_query("""
SELECT
    m.product,
    v.sale_month,
    v.revenue
FROM monthly_sales AS m
CROSS JOIN LATERAL (
    VALUES
        (DATE '2025-01-01', m.jan_revenue),
        (DATE '2025-02-01', m.feb_revenue),
        (DATE '2025-03-01', m.mar_revenue)
) AS v(sale_month, revenue)
WHERE v.revenue IS NOT NULL
ORDER BY m.product, v.sale_month;
""", title="Unpivot and drop NULL cells")


# Part X · Round Trips and Information Loss

A reshape is **reversible** only if enough information survives.

### Safe idea

```text
long product-month totals
    ↓ pivot
wide product row
    ↓ unpivot
long product-month totals
```

### Potentially lossy idea

```text
transaction-level sales
    ↓ SUM into product-month cell
transaction detail disappears
```

Once several source rows are aggregated into one cell, the individual transactions cannot be recovered from the pivoted result alone.


## 21. Pivot, then unpivot


In [ ]:
display_query("""
WITH wide AS (
    SELECT
        product,
        SUM(revenue) FILTER (
            WHERE sale_month = DATE '2025-01-01'
        ) AS jan_revenue,
        SUM(revenue) FILTER (
            WHERE sale_month = DATE '2025-02-01'
        ) AS feb_revenue,
        SUM(revenue) FILTER (
            WHERE sale_month = DATE '2025-03-01'
        ) AS mar_revenue
    FROM sales
    GROUP BY product
)
SELECT
    w.product,
    v.sale_month,
    v.revenue
FROM wide AS w
CROSS JOIN LATERAL (
    VALUES
        (DATE '2025-01-01', w.jan_revenue),
        (DATE '2025-02-01', w.feb_revenue),
        (DATE '2025-03-01', w.mar_revenue)
) AS v(sale_month, revenue)
ORDER BY w.product, v.sale_month;
""", title="Pivot then unpivot")


### What came back?

The round trip returns **product-month totals**.

It does not return the original transaction IDs.

So the reshape is reversible at the summarized grain, not at the original transaction grain.


# Part XI · Join Before Pivot Carefully

A join can multiply rows before the reshape.

If one patient has several admissions and another joined table also has several rows per patient, the combination can create more rows than expected.

Before pivoting:
1. know the grain of each input,
2. know the relationship cardinality,
3. inspect row counts after joins,
4. pre-aggregate when appropriate.


## 22. Patient admission counts by month


In [ ]:
display_query("""
SELECT
    p.subject_id,
    p.name,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-01-01'
    ) AS jan_admissions,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-02-01'
    ) AS feb_admissions,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-03-01'
    ) AS mar_admissions
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.subject_id = p.subject_id
GROUP BY p.subject_id, p.name
ORDER BY p.subject_id;
""", title="Patient-month admission matrix")


### Why start from patients?

The `LEFT JOIN` keeps patients who have no admissions.

That is a grain and completeness decision, not just a join-style preference.


# Part XII · Pre-Aggregate Before Reshaping

A strong pattern is:

```text
raw detailed rows
    ↓
pre-aggregate to meaningful long grain
    ↓
pivot
```

This separates:
- the aggregation decision,
- from the structural reshape.


## 23. Pre-aggregate product-month revenue, then pivot


In [ ]:
display_query("""
WITH monthly_product AS (
    SELECT
        product,
        category,
        sale_month,
        SUM(revenue) AS revenue
    FROM sales
    GROUP BY product, category, sale_month
)
SELECT
    category,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-01-01'
    ) AS jan_revenue,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-02-01'
    ) AS feb_revenue,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-03-01'
    ) AS mar_revenue
FROM monthly_product
GROUP BY category
ORDER BY category;
""", title="Pre-aggregate then pivot")


### Grain changes

| Stage | Grain |
|---|---|
| raw `sales` | one row per transaction |
| `monthly_product` CTE | one row per product-category-month |
| final result | one row per category |

Writing these grain statements before the SQL makes the query much easier to reason about.


# Part XIII · Dynamic Categories

Fixed pivot columns are **static**.

If you write:

```sql
... WHERE diagnosis = 'Pneumonia'
... WHERE diagnosis = 'Fracture'
... WHERE diagnosis = 'Flu'
```

and a new diagnosis appears tomorrow, the query does not automatically create a new column.

That is why you should inspect categories before designing a fixed pivot.


## 24. Discover categories first


In [ ]:
display_query("""
SELECT DISTINCT diagnosis
FROM admissions
ORDER BY diagnosis;
""", title="Available diagnoses")


In [ ]:
display_query("""
SELECT DISTINCT product
FROM sales
ORDER BY product;
""", title="Available products")


### Static vs. dynamic output

Static columns are often appropriate when:
- a report schema must stay consistent,
- known categories are required by downstream software,
- students are learning explicit pivot logic.

Dynamic SQL or extensions such as `crosstab` may help when categories are not known in advance, but they also make the workflow more complex.


# Part XIV · Choose Shape for the Next Task

Long and wide are not “good” and “bad.”

They support different downstream tasks.

### Long form is often convenient for:
- grouping,
- filtering,
- plotting,
- time-series analysis,
- adding new categories.

### Wide form is often convenient for:
- human-readable matrices,
- exports,
- feature tables,
- side-by-side comparisons,
- fixed report layouts.


## 25. Keep charting data long


In [ ]:
display_query("""
SELECT
    sale_month,
    product,
    SUM(revenue) AS revenue
FROM sales
GROUP BY sale_month, product
ORDER BY sale_month, product;
""", title="Long product-month result for charting")


The long result naturally contains:
- one categorical column: `product`
- one time column: `sale_month`
- one measure: `revenue`

That structure is often easier for visualization tools than a separate column for every product.


# Part XV · Validation

A reshape is not complete just because the query runs.

Useful checks include:
- source total vs. reshaped total,
- source row count vs. expected target row count,
- duplicate source rows per target cell,
- unexpected NULLs,
- category discovery,
- round-trip checks where appropriate.


## 26. Validate total revenue before reshaping


In [ ]:
display_query("""
SELECT
    SUM(revenue) AS source_total_revenue
FROM sales;
""", title="Source total revenue")


## 27. Validate revenue at product-month grain


In [ ]:
display_query("""
SELECT
    product,
    sale_month,
    COUNT(*) AS transaction_count,
    SUM(revenue) AS total_revenue
FROM sales
GROUP BY product, sale_month
ORDER BY product, sale_month;
""", title="Product-month validation")


## 28. Find cells that combine multiple source rows


In [ ]:
display_query("""
SELECT
    product,
    sale_month,
    COUNT(*) AS source_rows
FROM sales
GROUP BY product, sale_month
HAVING COUNT(*) > 1
ORDER BY product, sale_month;
""", title="Potential aggregation cells")


## 29. Check NULLs in the wide table


In [ ]:
display_query("""
SELECT
    product,
    jan_revenue,
    feb_revenue,
    mar_revenue
FROM monthly_sales
WHERE jan_revenue IS NULL
   OR feb_revenue IS NULL
   OR mar_revenue IS NULL
ORDER BY product;
""", title="Wide rows containing missing cells")


## 30. Compare row counts before and after unpivot


In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM monthly_sales) AS wide_rows,

    (
        SELECT COUNT(*)
        FROM monthly_sales AS m
        CROSS JOIN LATERAL (
            VALUES
                (DATE '2025-01-01', m.jan_revenue),
                (DATE '2025-02-01', m.feb_revenue),
                (DATE '2025-03-01', m.mar_revenue)
        ) AS v(sale_month, revenue)
    ) AS long_rows_including_nulls,

    (
        SELECT COUNT(*)
        FROM monthly_sales AS m
        CROSS JOIN LATERAL (
            VALUES
                (DATE '2025-01-01', m.jan_revenue),
                (DATE '2025-02-01', m.feb_revenue),
                (DATE '2025-03-01', m.mar_revenue)
        ) AS v(sale_month, revenue)
        WHERE v.revenue IS NOT NULL
    ) AS long_rows_excluding_nulls;
""", title="Wide vs long row counts")


### Interpret the row count

If the wide table has 4 products and 3 month columns, a complete unpivot can produce:

```text
4 × 3 = 12 rows
```

Dropping NULL cells can reduce that number.

That row-count change should be intentional.


# Part XVI · Applied Examples


## 31. Student grade matrix


In [ ]:
display_query("""
SELECT
    s.student_id,
    s.student_name,
    MAX(g.grade) FILTER (
        WHERE g.course_code = 'CSC101'
    ) AS csc101,
    MAX(g.grade) FILTER (
        WHERE g.course_code = 'STA201'
    ) AS sta201,
    MAX(g.grade) FILTER (
        WHERE g.course_code = 'DBA301'
    ) AS dba301
FROM students AS s
LEFT JOIN grades AS g
    ON g.student_id = s.student_id
GROUP BY s.student_id, s.student_name
ORDER BY s.student_id;
""", title="Student grade matrix")


Output grain:

> **one row per student**

A missing course remains `NULL`, preserving the difference between “no recorded grade” and a numeric grade.


## 32. Patient-month matrix with total admissions


In [ ]:
display_query("""
SELECT
    p.subject_id,
    p.name,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-01-01'
    ) AS jan,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-02-01'
    ) AS feb,
    COUNT(a.hadm_id) FILTER (
        WHERE DATE_TRUNC('month', a.admit_time) = DATE '2025-03-01'
    ) AS mar,
    COUNT(a.hadm_id) AS total_admissions
FROM patients AS p
LEFT JOIN admissions AS a
    ON a.subject_id = p.subject_id
GROUP BY p.subject_id, p.name
ORDER BY total_admissions DESC, p.subject_id;
""", title="Patient admission matrix")


# Part XVII · Practice

For each problem, write these two statements before writing SQL:

```text
Source grain: ...
Target grain: ...
```

Then decide:
- aggregation rule,
- NULL policy,
- categories,
- validation check.


## Practice 1 · Grade matrix

Create one row per student with columns:
- `CSC101`
- `STA201`
- `DBA301`

Keep missing courses as `NULL`.

<details>
<summary>Hint</summary>

Start from `students` and `LEFT JOIN grades`.

Use conditional aggregation for the course columns.

</details>


In [ ]:
practice_1 = """
-- Source grain:
-- Target grain:

-- Write your SQL here


"""


## Practice 2 · Patient-month admissions

Create one row per patient with:
- January admissions
- February admissions
- March admissions
- total admissions

Preserve patients with zero admissions.

<details>
<summary>Hint</summary>

Start from `patients`.

Use a `LEFT JOIN` to `admissions`.

</details>


In [ ]:
practice_2 = """
-- Source grain:
-- Target grain:

-- Write your SQL here


"""


## Practice 3 · Survey pivot

Create one row per respondent with:
- Q1
- Q2
- Q3

Keep unanswered questions as `NULL`.


In [ ]:
practice_3 = """
-- Source grain:
-- Target grain:

-- Write your SQL here


"""


## Practice 4 · Monthly sales unpivot

Convert `monthly_sales` into:

```text
product, sale_month, revenue
```

Remove rows where revenue is `NULL`.

Try both:
1. `UNION ALL`
2. `LATERAL VALUES`


In [ ]:
practice_4 = """
-- Write your SQL here


"""


## Practice 5 · Diagnosis counts by month

Create one row per month with columns:
- Pneumonia
- Fracture
- Flu

Use either:
- `COUNT(*) FILTER`
- or `SUM(CASE ...)`


In [ ]:
practice_5 = """
-- Write your SQL here


"""


# Part XVIII · Assessment Worked Example

### Task

Reshape the long `sales` table to one row per product.

Return:
- January revenue
- February revenue
- March revenue

Preserve `NULL` when a product has no observation for a month.

### Reasoning

Source grain:

> one row per transaction

Intermediate useful grain:

> one row per product-month

Final grain:

> one row per product

Because several transactions may contribute to one product-month cell, use `SUM`.


In [ ]:
display_query("""
SELECT
    product,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-01-01'
    ) AS jan_revenue,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-02-01'
    ) AS feb_revenue,
    SUM(revenue) FILTER (
        WHERE sale_month = DATE '2025-03-01'
    ) AS mar_revenue
FROM sales
GROUP BY product
ORDER BY product;
""", title="Assessment sample solution")


# Part XIX · Quick Concept Check

### 1. What is the first question before reshaping?
<details><summary>Show answer</summary>What is the current grain, and what should the target grain be?</details>

### 2. Why might a pivot need aggregation?
<details><summary>Show answer</summary>Several source rows may map to the same output cell.</details>

### 3. When are NULL and zero different?
<details><summary>Show answer</summary>When NULL means “no observation” while zero is an actual observed value.</details>

### 4. What does `LATERAL VALUES` help do?
<details><summary>Show answer</summary>Generate several long-form rows from each wide source row.</details>

### 5. Why can pivoting be lossy?
<details><summary>Show answer</summary>Aggregation can combine several detailed rows into one value, discarding row-level detail.</details>

### 6. Why discover categories before a static pivot?
<details><summary>Show answer</summary>Fixed output columns only include categories explicitly written into the query.</details>

### 7. What should you validate after a reshape?
<details><summary>Show answer</summary>Totals, row counts, grain, NULL behavior, categories, and unexpected duplication.</details>


# Module 10 Summary

### Pivot
Long rows become columns.

### Unpivot
Wide columns become rows.

### Grain
Always state it before and after reshaping.

### Conditional aggregation
Use `CASE` or PostgreSQL `FILTER` to create pivot columns.

### NULL semantics
Do not turn missing data into zero unless the meaning is truly equivalent.

### LATERAL VALUES
A compact PostgreSQL pattern for converting repeated wide columns into rows.

### Validation
A reshape is not complete until you verify:
- totals,
- row counts,
- categories,
- NULLs,
- aggregation rules,
- output grain.

### Decision guide

| Question | Direction |
|---|---|
| Need categories as separate columns? | pivot wider |
| Need repeated columns as values in rows? | unpivot longer |
| Need plotting/filtering across categories? | long often works better |
| Need fixed report matrix? | wide may work better |
| Multiple rows map to one cell? | aggregate before/during pivot |
| New categories can appear? | inspect categories; static pivot may need maintenance |

## Next
**Module 11: Advanced SQL — CTEs & Window Functions**
